In [1]:
# Insumos requeridos:
%pip install streamlit pandas transformers accelerate

  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
   ---------------------------------------- 0.0/11.2 MB ? eta -:--:--
   ------------------------ --------------- 6.8/11.2 MB 37.9 MB/s eta 0:00:01
   ---------------------------------------- 11.2/11.2 MB 37.8 MB/s  0:00:00
   ---------------------------------------- 0.0/797.2 kB ? eta -:--:--
   ---------------------------------------- 797.2/797.2 kB 31.1 MB/s  0:00:00
Using cached itsdangerous-2.2.0-py3-none-any.whl (16 kB)
   ---------------------------------------- 0.0/11.4 MB ? eta -:--:--
   ------------------------------------ --- 10.5/11.4 MB 51.9 MB/s eta 0:00:01
   ---------------------------------------- 11.4/11.4 MB 47.2 MB/s  0:00:00

   ---------------------------------------- 0/9 [websockets]
   ---- ----------------------------------- 1/9 [watchdog]
   -------- ------------------------------- 2/9 [uvicorn]
   -------- ------------------------------- 2/9 [uvicorn]
   ---------------------- -------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


### Creación de app

In [ ]:
%%writefile app.py
import io
import time

import pandas as pd
import streamlit as st
import torch
from transformers import pipeline

# ======================= Configuración =======================
MODELO_TEXTO = "Qwen/Qwen2.5-1.5B-Instruct"          # chatbot
MODELO_IMAGEN = "black-forest-labs/FLUX.2-klein-4B"   # imágenes
HAY_GPU = torch.cuda.is_available()
MAX_FILAS_COMPLETAS = 60   # si la tabla tiene más filas, al modelo solo se le envía una muestra
MENSAJES_MEMORIA = 6       # cuántos mensajes previos recuerda el chatbot

MODO_CHAT = "💬 Chatear (texto o tabla)"
MODO_IMAGEN = "🎨 Generar imagen"

st.set_page_config(page_title="Chatbot Hugging Face", page_icon="🤖", layout="wide")


# ======================= Carga de modelos =======================
# Streamlit vuelve a ejecutar todo el script en cada interacción.
# @st.cache_resource hace que cada modelo se cargue UNA sola vez y se reutilice.
@st.cache_resource(show_spinner="Cargando modelo de texto (solo la primera vez)...")
def cargar_modelo_texto():
    return pipeline(
        "text-generation",
        model=MODELO_TEXTO,
        dtype=torch.bfloat16 if HAY_GPU else torch.float32,
        device_map="auto",
    )


@st.cache_resource(show_spinner="Cargando FLUX.2 [klein] 4B (solo la primera vez, tarda un poco)...")
def cargar_modelo_imagen():
    from diffusers import Flux2KleinPipeline
    from diffusers.quantizers import PipelineQuantizationConfig

    # Cuantización a 4 bits para que quepa en 8 GB de VRAM
    quant_config = PipelineQuantizationConfig(
        quant_backend="bitsandbytes_4bit",
        quant_kwargs={
            "load_in_4bit": True,
            "bnb_4bit_quant_type": "nf4",
            "bnb_4bit_compute_dtype": torch.bfloat16,
        },
        components_to_quantize=["transformer", "text_encoder"],
    )
    pipe = Flux2KleinPipeline.from_pretrained(
        MODELO_IMAGEN, torch_dtype=torch.bfloat16, quantization_config=quant_config
    )
    pipe.enable_model_cpu_offload()
    return pipe


# ======================= Funciones auxiliares =======================
def leer_csv(archivo):
    """Lee un CSV detectando automáticamente el separador (, o ;) y la codificación."""
    for codificacion in ("utf-8", "latin-1"):
        try:
            archivo.seek(0)
            return pd.read_csv(archivo, sep=None, engine="python", encoding=codificacion)
        except UnicodeDecodeError:
            continue
    raise ValueError("No se pudo leer el archivo CSV.")


def describir_tabla(df):
    """Convierte la tabla en texto que el modelo de lenguaje pueda leer."""
    partes = [
        f"La tabla tiene {df.shape[0]} filas y {df.shape[1]} columnas.",
        "Columnas (tipo de dato):\n" + "\n".join(f"- {c} ({t})" for c, t in df.dtypes.astype(str).items()),
    ]
    if len(df) <= MAX_FILAS_COMPLETAS:
        partes.append("Datos completos (CSV):\n" + df.to_csv(index=False))
    else:
        partes.append(
            f"Primeras {MAX_FILAS_COMPLETAS} filas de {len(df)} (CSV):\n"
            + df.head(MAX_FILAS_COMPLETAS).to_csv(index=False)
        )
    # Los modelos pequeños se equivocan haciendo cuentas: les damos los cálculos hechos con pandas
    numericas = df.select_dtypes("number")
    if not numericas.empty:
        partes.append("Suma total de cada columna numérica (calculada con pandas):\n" + numericas.sum().round(2).to_string())
    partes.append("Estadísticas descriptivas (calculadas con pandas):\n" + df.describe(include="all").round(2).to_string())
    return "\n\n".join(partes)


def responder_texto(historial, df=None):
    """Genera la respuesta del chatbot con el modelo de texto de Hugging Face."""
    chat = cargar_modelo_texto()

    sistema = "Eres un asistente útil y amable. Responde siempre en español, de forma clara y concisa."
    if df is not None:
        sistema += (
            "\n\nEl usuario subió una tabla de datos. Responde sus preguntas usando ÚNICAMENTE "
            "la información de esta tabla. Si la respuesta no está en la tabla, dilo.\n\n"
            + describir_tabla(df)
        )

    # Solo los últimos mensajes de texto (este modelo no entiende imágenes)
    previos = [
        {"role": m["rol"], "content": m["contenido"]}
        for m in historial
        if m["tipo"] == "texto"
    ][-MENSAJES_MEMORIA:]

    salida = chat(
        [{"role": "system", "content": sistema}] + previos,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.2 if df is not None else 0.7,  # más preciso cuando responde sobre datos
    )
    return salida[0]["generated_text"][-1]["content"]


def generar_imagen(prompt, lado, pasos, semilla):
    """Genera una imagen con FLUX.2 [klein] 4B."""
    pipe = cargar_modelo_imagen()
    imagen = pipe(
        prompt=prompt,
        width=lado,
        height=lado,
        num_inference_steps=pasos,
        guidance_scale=1.0,
        generator=torch.Generator("cuda" if HAY_GPU else "cpu").manual_seed(semilla),
    ).images[0]
    if HAY_GPU:
        torch.cuda.empty_cache()
    return imagen


def imagen_a_png(imagen):
    buffer = io.BytesIO()
    imagen.save(buffer, format="PNG")
    return buffer.getvalue()


# ======================= Estado de la conversación =======================
if "mensajes" not in st.session_state:
    st.session_state.mensajes = []


# ======================= Barra lateral =======================
with st.sidebar:
    st.header("⚙️ Opciones")
    modo = st.radio("¿Qué quieres hacer?", [MODO_CHAT, MODO_IMAGEN])

    st.divider()
    st.subheader("📊 Tabla de datos")
    archivo = st.file_uploader("Sube un archivo CSV", type=["csv"])
    df = None
    if archivo is not None:
        try:
            df = leer_csv(archivo)
            st.success(f"Tabla cargada: {df.shape[0]} filas × {df.shape[1]} columnas")
        except Exception as e:
            st.error(f"No se pudo leer el CSV: {e}")
    usar_tabla = st.checkbox("Responder usando la tabla", value=True, disabled=df is None)

    if modo == MODO_IMAGEN:
        st.divider()
        st.subheader("🎨 Imagen")
        lado = st.select_slider("Tamaño (px)", options=[512, 768, 1024], value=768)
        pasos = st.slider("Pasos de generación", min_value=1, max_value=8, value=4)
        semilla = st.number_input("Semilla", value=42, step=1)

    st.divider()
    if st.button("🗑️ Borrar conversación"):
        st.session_state.mensajes = []
        st.rerun()
    st.caption(
        f"Texto: `{MODELO_TEXTO}`  \nImagen: `{MODELO_IMAGEN}`  \n"
        f"Dispositivo: {'GPU (' + torch.cuda.get_device_name(0) + ')' if HAY_GPU else 'CPU'}"
    )


# ======================= Página principal =======================
st.title("🤖 Chatbot con modelos de Hugging Face")
st.caption("Chatea, haz preguntas sobre una tabla CSV o genera imágenes. Todo corre localmente en tu computadora.")

if df is not None:
    with st.expander("👀 Vista previa de la tabla"):
        st.dataframe(df)

# Mostrar el historial de la conversación
for i, m in enumerate(st.session_state.mensajes):
    with st.chat_message(m["rol"]):
        if m["tipo"] == "imagen":
            st.image(m["contenido"], caption=m.get("info"))
            st.download_button(
                "⬇️ Descargar PNG",
                data=imagen_a_png(m["contenido"]),
                file_name=f"imagen_{i}.png",
                mime="image/png",
                key=f"descarga_{i}",
            )
        else:
            st.markdown(m["contenido"])
            if m.get("info"):
                st.caption(m["info"])

# Entrada del usuario
ayuda = "Describe la imagen que quieres generar..." if modo == MODO_IMAGEN else "Escribe tu mensaje o una pregunta sobre la tabla..."
prompt = st.chat_input(ayuda)

if prompt:
    # Los prompts de imagen se guardan con otro tipo para no mezclarlos con la memoria del chatbot
    tipo_usuario = "prompt_imagen" if modo == MODO_IMAGEN else "texto"
    st.session_state.mensajes.append({"rol": "user", "tipo": tipo_usuario, "contenido": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        inicio = time.time()
        try:
            if modo == MODO_IMAGEN:
                with st.spinner("Generando imagen..."):
                    imagen = generar_imagen(prompt, lado, pasos, int(semilla))
                info = f"FLUX.2 [klein] 4B · {lado}×{lado} px · {pasos} pasos · ⏱️ {time.time() - inicio:.1f} s"
                st.session_state.mensajes.append({"rol": "assistant", "tipo": "imagen", "contenido": imagen, "info": info})
            else:
                tabla = df if (df is not None and usar_tabla) else None
                with st.spinner("Pensando..."):
                    respuesta = responder_texto(st.session_state.mensajes, tabla)
                info = f"{MODELO_TEXTO}{' · usando la tabla' if tabla is not None else ''} · ⏱️ {time.time() - inicio:.1f} s"
                st.session_state.mensajes.append({"rol": "assistant", "tipo": "texto", "contenido": respuesta, "info": info})
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            st.session_state.mensajes.append({
                "rol": "assistant",
                "tipo": "aviso",
                "contenido": "⚠️ La GPU se quedó sin memoria. Prueba con un tamaño de imagen menor (512 px).",
            })
    st.rerun()


Writing app.py


### Inicialización de la app

In [4]:
import subprocess, sys

# Streamlit corre en un proceso aparte; el notebook queda libre mientras tanto
app = subprocess.Popen([
    sys.executable, "-m", "streamlit", "run", "app.py",
    "--server.headless", "true",               # evita que Streamlit se quede esperando una respuesta en la terminal
    "--browser.gatherUsageStats", "false",
])
print("App iniciada. Abre en tu navegador: http://localhost:8501")


App iniciada. Abre en tu navegador: http://localhost:8501


### Para detener la app

In [5]:
app.terminate()
print("App detenida.")

App detenida.
